# 02 · Quantile Forecasts (P10 / P50 / P90 …)

Instead of a full distribution we forecast a **set of quantiles** $\{\hat q_\tau : \tau\in\mathcal T\}$, e.g.
$\mathcal T=\{0.1,0.5,0.9\}$. This is the dominant format in retail/supply-chain (Amazon's P50/P90), energy
(GEFCom) and the M5 uncertainty competition.

## Theory

### Quantile function
$q_\tau = F^{-1}(\tau)=\inf\{y: F(y)\ge\tau\}$. "P90 = 1 200 units" means $\Pr(Y\le 1200)=0.9$.

### Pinball (quantile) loss, and why it works
$$\rho_\tau(u)=\begin{cases}\tau\,u & u\ge 0\\ (\tau-1)\,u & u<0\end{cases},\qquad u=y-q.$$
Under-prediction costs $\tau$ per unit, over-prediction costs $1-\tau$ per unit.
Minimise the expected loss: $\frac{d}{dq}\mathbb E[\rho_\tau(Y-q)] = -\tau(1-F(q)) + (1-\tau)F(q) = F(q)-\tau = 0
\Rightarrow q=F^{-1}(\tau)$.
So **any model trained with pinball loss estimates the $\tau$-quantile directly, with no distributional assumption**.

### Ways to get quantile forecasts
| Approach | How | Pros / cons |
|---|---|---|
| From a parametric distribution | $\hat q_\tau=\hat\mu+\hat\sigma\,z_\tau$ | coherent, needs correct distribution |
| **Quantile regression** (Koenker & Bassett 1978) | minimise $\sum_t\rho_\tau(y_t-x_t^\top\beta_\tau)$ separately for each $\tau$ | assumption-free, each $\tau$ has its own slopes, so it handles heteroscedasticity; can **cross** |
| Empirical residual quantiles | point model + quantiles of past errors | simple, assumes errors are i.i.d. |
| ML with quantile loss | GBM (`objective='quantile'`), NN multi-quantile heads (MQ-RNN, TFT) | flexible, needs data, can cross |
| Quantile Regression Averaging | QR on the point forecasts of several models | good model-combination trick |

### Quantile crossing
Separately estimated quantiles may violate $\hat q_{0.8}<\hat q_{0.9}$. Fixes: **monotone rearrangement**
(sort the quantiles; Chernozhukov et al. 2010, it never increases pinball loss), non-crossing constraints, or predicting
increments $\hat q_{\tau_{k+1}}=\hat q_{\tau_k}+\text{softplus}(\cdot)$.

### Evaluation
* Pinball loss per $\tau$ (proper for that quantile)
* **Weighted / scaled quantile loss** $\text{wQL}_\tau = 2\sum_t\rho_\tau(y_t-\hat q_{\tau,t})/\sum_t|y_t|$, averaged over $\tau$
* **Quantile calibration**: the fraction of $y_t\le\hat q_\tau$ should be $\approx\tau$
* $2\times$ mean pinball over a dense grid of $\tau$ ≈ CRPS

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.graph_objects as go
from scipy import stats
import statsmodels.api as sm
from statsmodels.regression.quantile_regression import QuantReg
from statsmodels.tsa.statespace.sarimax import SARIMAX
import pf_utils as pf
pf.set_style()
rng = np.random.default_rng(2)
TAUS = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]

## 1. Toy example: why quantile regression handles heteroscedasticity

Monthly sales vs marketing spend where the noise **grows with spend and is right-skewed**.
OLS + a Gaussian band assumes constant spread; quantile regression fits a separate line per $\tau$.

In [2]:
n = 240
x = rng.uniform(0, 100, n)
y = 200 + 3 * x + (5 + 0.4 * x) * (rng.gamma(2.0, 1.0, n) - 2.0)
X = sm.add_constant(x)
xs = np.linspace(0, 100, 50); Xs = sm.add_constant(xs)

ols = sm.OLS(y, X).fit()
fig = pf.subplots(1, 2, titles=["OLS mean ± Gaussian quantiles (constant width)", "Quantile regression (one line per τ)"],
                  shared_yaxes=True, height=420)
for c in (1, 2):
    fig.add_trace(go.Scatter(x=x, y=y, mode="markers", name="observations", showlegend=c == 1,
                             marker=dict(color=pf.INK["neutral"], size=8, opacity=0.6),
                             hovertemplate="spend %{x:.0f} → sales %{y:.0f}<extra></extra>"), row=1, col=c)
for k, tau in enumerate([0.05, 0.5, 0.95]):
    qr = QuantReg(y, X).fit(q=tau)
    fig.add_trace(go.Scatter(x=xs, y=ols.predict(Xs) + stats.norm.ppf(tau) * np.sqrt(ols.scale), mode="lines",
                             name=f"OLS-Gaussian τ={tau}", line=dict(color=pf.SERIES[k], width=2, dash="dash")), row=1, col=1)
    fig.add_trace(go.Scatter(x=xs, y=qr.predict(Xs), mode="lines", name=f"QR τ={tau} (slope {qr.params[1]:.2f})",
                             line=dict(color=pf.SERIES[k], width=2)), row=1, col=2)
fig.update_xaxes(title_text="marketing spend")
fig.show()

for tau in [0.05, 0.5, 0.95]:
    q_ols = ols.fittedvalues + stats.norm.ppf(tau) * np.sqrt(ols.scale)
    q_qr = QuantReg(y, X).fit(q=tau).fittedvalues
    print(f"τ={tau}:  share below  OLS-Gaussian={np.mean(y <= q_ols):.2f}   QR={np.mean(y <= q_qr):.2f}")

τ=0.05:  share below  OLS-Gaussian=0.02   QR=0.05
τ=0.5:  share below  OLS-Gaussian=0.55   QR=0.50
τ=0.95:  share below  OLS-Gaussian=0.92   QR=0.95


The OLS-Gaussian band is **symmetric and constant-width**: it misses the skew (see the shares above) and is too wide
at low spend and too narrow at high spend.
QR's slopes differ by $\tau$. That **is** the conditional distribution changing shape.

## 2. Monthly retail sales: three ways to produce P5…P95

* Train: 2015–2022 (96 months), test: 2023–2024 (24 months, horizons 1–24)
* Features for regression approaches: trend + month dummies, on $\log y$ (quantiles survive the exp back-transform)

In [3]:
y = pf.load_retail_total()
train, test = y[:"2022-12-01"], y["2023-01-01":]
H = len(test)
Xtr = pf.seasonal_design(train.index, origin="2015-01-01")
Xte = pf.seasonal_design(test.index, origin="2015-01-01")
ltr = np.log(train)

Q = {}  # method -> {tau: array}

# (a) parametric: SARIMA on log y -> Gaussian quantiles -> exp
res = SARIMAX(ltr, order=(1, 0, 0), seasonal_order=(0, 1, 1, 12), trend="c").fit(disp=False)
fc = res.get_forecast(H)
m, s = np.asarray(fc.predicted_mean), np.sqrt(np.asarray(fc.var_pred_mean))
Q["SARIMA (parametric)"] = {t: np.exp(m + s * stats.norm.ppf(t)) for t in TAUS}

# (b) linear quantile regression, one model per tau
Q["Quantile regression"] = {t: np.exp(QuantReg(ltr, Xtr).fit(q=t, max_iter=5000).predict(Xte).to_numpy()) for t in TAUS}

# (c) OLS point model + empirical quantiles of in-sample residuals
ols = sm.OLS(ltr, Xtr).fit()
Q["OLS + empirical residuals"] = {t: np.exp(ols.predict(Xte).to_numpy() + np.quantile(ols.resid, t)) for t in TAUS}

fig = pf.subplots(1, 3, titles=list(Q), shared_yaxes=True, height=440)
for c, q in enumerate(Q.values(), start=1):
    pf.add_fan(fig, test.index, q, history=train, actual=test.values, n_history=36, row=1, col=c,
               showlegend=c == 1, fmt=",.0f")
fig.update_layout(legend=dict(orientation="h", yanchor="top", y=-0.12, x=0))
fig.show()

## 3. Evaluation: pinball loss, wQL, quantile calibration

In [9]:
yt = test.to_numpy()
rows = {}
for name, q in Q.items():
    r = {f"pinball τ={t}": pf.pinball_loss(yt, q[t], t).mean() for t in TAUS}
    r["mean wQL"] = pf.weighted_quantile_loss(yt, q)
    r["CRPS≈2·mean pinball"] = pf.crps_quantiles(yt, q).mean()
    r["90% coverage (P5–P95)"] = pf.coverage(yt, q[0.05], q[0.95])
    rows[name] = r
pd.DataFrame(rows).T

,pinball τ=0.05,pinball τ=0.1,pinball τ=0.25,pinball τ=0.5,pinball τ=0.75,pinball τ=0.9,pinball τ=0.95,mean wQL,CRPS≈2·mean pinball,90% coverage (P5–P95)
SARIMA (parametric),8.578,14.553,33.345,51.468,42.642,22.750,13.657,0.027,53.427,0.958
Quantile regression,14.183,20.410,58.200,65.949,38.415,20.224,10.125,0.032,65.002,0.792
OLS + empirical residuals,9.628,20.354,42.228,59.590,42.088,21.523,12.687,0.030,59.457,0.833


In [10]:
fig = pf.subplots(1, 2, titles=["Quantile calibration (test, 24 months)", "Width of P5–P95 by horizon"], height=400)
fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="perfect calibration", hoverinfo="skip",
                         line=dict(color=pf.INK["muted"], dash="dash", width=1)), row=1, col=1)
h = np.arange(1, H + 1)
for k, (name, q) in enumerate(Q.items()):
    fig.add_trace(go.Scatter(x=TAUS, y=[np.mean(yt <= q[t]) for t in TAUS], mode="lines+markers", name=name,
                             legendgroup=name, line=dict(color=pf.SERIES[k], width=2), marker=dict(size=8),
                             hovertemplate=name + "<br>τ=%{x}: share below %{y:.2f}<extra></extra>"), row=1, col=1)
    fig.add_trace(go.Scatter(x=h, y=q[0.95] - q[0.05], mode="lines", name=name, legendgroup=name, showlegend=False,
                             line=dict(color=pf.SERIES[k], width=2),
                             hovertemplate=name + "<br>h=%{x}: width %{y:,.0f}<extra></extra>"), row=1, col=2)
fig.update_xaxes(title_text="nominal τ", row=1, col=1)
fig.update_yaxes(title_text="observed share y ≤ q_τ", row=1, col=1)
fig.update_xaxes(title_text="horizon (months)", row=1, col=2)
fig.show()

**Interpretation**
* SARIMA uses the latest observations (AR term), so it is sharper at short horizons and widens with $h$.
* The regression approaches use only deterministic features (trend + month), so their width **does not grow with $h$**.
  It only follows the seasonal level (the log-scale spread is multiplied by the level) and, for QR, the month-specific
  quantile coefficients. Those are estimated from 8 observations per month, which is why the QR width jumps around.
  They estimate the *marginal* spread around the seasonal trend, and they cannot exploit the persistence of shocks.
  For long horizons this can be fine. For short horizons add lagged values (direct strategy: one model per horizon).
* With only 24 test points, calibration curves are noisy. Use a rolling backtest (nb 03, 12) before concluding anything.

## 4. Quantile crossing and monotone rearrangement
With little data and many quantile levels, independently fitted QR lines can cross.

In [11]:
small = train[-30:]                       # deliberately tiny training sample
Xs_ = pf.seasonal_design(small.index, origin="2015-01-01")
dense = np.round(np.arange(0.05, 0.96, 0.05), 2)
qs = np.column_stack([QuantReg(np.log(small), Xs_).fit(q=t, max_iter=5000).predict(Xte).to_numpy() for t in dense])
crossings = np.sum(np.diff(qs, axis=1) < 0, axis=1)
print("months with ≥1 crossing:", np.sum(crossings > 0), "of", H)

qs_sorted = np.sort(qs, axis=1)           # monotone rearrangement
pl_raw = np.mean([pf.pinball_loss(np.log(yt), qs[:, k], t).mean() for k, t in enumerate(dense)])
pl_sorted = np.mean([pf.pinball_loss(np.log(yt), qs_sorted[:, k], t).mean() for k, t in enumerate(dense)])
print(f"mean pinball (log scale)  raw={pl_raw:.5f}   rearranged={pl_sorted:.5f}   (rearranging never hurts)")

k = int(np.argmax(crossings))
fig = go.Figure()
fig.add_trace(go.Scatter(x=dense, y=qs[k], mode="lines+markers", name="raw QR quantiles (they cross)",
                         line=dict(color=pf.SERIES[1], width=2), marker=dict(size=8)))
fig.add_trace(go.Scatter(x=dense, y=qs_sorted[k], mode="lines+markers", name="rearranged (sorted)",
                         line=dict(color=pf.SERIES[0], width=2, dash="dash"), marker=dict(size=8, symbol="square")))
fig.update_layout(title=f"Quantile curve for {test.index[k]:%b %Y}: a valid quantile curve never goes down",
                  height=380, hovermode="x unified", xaxis_title="τ", yaxis_title="log sales")
fig.show()

months with ≥1 crossing: 24 of 24
mean pinball (log scale)  raw=0.02277   rearranged=0.02276   (rearranging never hurts)


## 5. Practical guidance

* Choose $\tau$ from the **decision**: newsvendor-optimal stock is the $\tau^*=c_u/(c_u+c_o)$ quantile (nb 12).
* For hierarchical / aggregated quantities, **quantiles do not add up**: $q_{0.9}(A+B)\ne q_{0.9}(A)+q_{0.9}(B)$.
  You need samples / joint distributions (nb 05, 09, 10).
* Gradient boosting with quantile loss (LightGBM `objective="quantile", alpha=τ`) is the workhorse for
  many-series retail problems. Tree models cannot extrapolate trends, so model growth rates or detrend first.
* Always check crossing, coverage per $\tau$ and pinball by horizon.

### Exercises
1. Add `log y_{t-12}` as a feature to the QR model (valid for $h\le12$) and re-score on 2023.
2. Implement Quantile Regression Averaging: regress $y$ on the point forecasts of SARIMA and OLS with QR.
3. Compute pinball loss by horizon bucket (1–6, 7–12, 13–24) for each method.